Установка зависимости

In [ ]:
!pip install PytorchWildlife
!pip install scikit-learn matplotlib Pillow tqdm

Подключение библиотек

In [ ]:
import os
import json
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Автовыбор устройства: MPS (Mac) -> CUDA (GPU) -> CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Используется MPS (Apple Silicon)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("✅ Используется CUDA GPU")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Используется CPU")

Гиперпараметры

In [ ]:
# =============================================
# ОБЯЗАТЕЛЬНО ПОМЕНЯЙТЕ ЭТИ ПУТИ ПОД СВОЙ ПК
# =============================================
DATASET_DIR  = "/Users/elvsevolod/PycharmProjects/SBER_animal_classifier/dataset"      # папка, где лежат jpg и json
JSON_FILE    = os.path.join(DATASET_DIR, "annotations.json")  # ваш json файл
CROPS_DIR    = DATASET_DIR   # <-- вот это изменение
MODEL_SAVE   = "resnet50_wildlife_best.pth"

# Классы: 0 = fox, 1 = wolf, 2 = other
CLASSES = {"fox": 0, "wolf": 1, "other": 2}
IDX2CLASS = {v: k for k, v in CLASSES.items()}
NUM_CLASSES = len(CLASSES)

# Гиперпараметры (рекомендуемые из статьи PytorchWildlife)
EPOCHS      = 60
BATCH_SIZE  = 32   # уменьшите до 16 если не хватает памяти на MPS
LR          = 0.01
IMG_SIZE    = 256
STEP_SIZE   = 20   # каждые 20 эпох lr уменьшается
GAMMA       = 0.1

Детекция через MegaDetector и сохранение кропов

In [ ]:
# import cv2
# from PytorchWildlife.models import detection as pw_detection
#
# with open(JSON_FILE, "r", encoding="utf-8") as f:
#     annotations = json.load(f)
#
# detector = pw_detection.MegaDetectorV6(
#     version="MDV6-yolov10-c",
#     device="cpu",
#     pretrained=True
# )
# # Порог уверенности задаём через атрибут, а не через аргумент функции
# detector.predictor.args.conf = 0.2
#
# os.makedirs(CROPS_DIR, exist_ok=True)
# for cls_name in CLASSES:
#     os.makedirs(os.path.join(CROPS_DIR, cls_name), exist_ok=True)
#
# skipped = 0
# saved = 0
#
# for item in tqdm(annotations, desc="🔍 Детекция и кроппинг"):
#     filename = item["filename"]
#     label    = item["label"]
#
#     if label not in CLASSES:
#         label = "other"
#
#     img_path = os.path.join(DATASET_DIR, filename)
#     if not os.path.exists(img_path):
#         skipped += 1
#         continue
#
#     img = np.array(Image.open(img_path).convert("RGB"))
#
#     # ✅ Только img — никаких лишних аргументов
#     results    = detector.single_image_detection(img)
#     detections = results.get("detections")
#
#     if detections is None or len(detections) == 0:
#         crop = Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE))
#         crop.save(os.path.join(CROPS_DIR, label, f"full_{filename}"))
#         saved += 1
#         continue
#
#     h, w = img.shape[:2]
#     for i, box in enumerate(detections.xyxy):
#         x1, y1, x2, y2 = map(int, box[:4])
#         pad = 10
#         x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
#         x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
#
#         crop_arr = img[y1:y2, x1:x2]
#         if crop_arr.size == 0:
#             continue
#
#         crop = Image.fromarray(crop_arr).resize((IMG_SIZE, IMG_SIZE))
#         crop.save(os.path.join(CROPS_DIR, label, f"crop{i}_{filename}"))
#         saved += 1
#
# print(f"\n✅ Сохранено кропов: {saved} | Пропущено файлов: {skipped}")


PyTorch Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import json

class WildlifeDatasetFromJSON(Dataset):
    def __init__(self, samples, dataset_dir, transform=None):
        """
        samples: список (filename, label_str) из JSON
        dataset_dir: папка, где лежат все jpg
        """
        self.dataset_dir = dataset_dir
        self.transform   = transform
        self.samples     = []  # список (полный_путь, числовой_класс)

        for filename, label_str in samples:
            # Если метка не в словаре классов — относим к "other"
            cls_idx = CLASSES.get(label_str, CLASSES["other"])
            full_path = os.path.join(dataset_dir, filename)
            if os.path.exists(full_path):
                self.samples.append((full_path, cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Загружаем JSON ─────────────────────────────────────────────
with open(JSON_FILE, "r", encoding="utf-8") as f:
    annotations = json.load(f)

# Превращаем JSON в список (filename, label)
all_samples = [(item["filename"], item["label"]) for item in annotations]
all_label_ints = [CLASSES.get(item["label"], CLASSES["other"]) for item in annotations]

print(f"📦 Всего записей в JSON: {len(all_samples)}")
for cls_name, cls_idx in CLASSES.items():
    count = sum(1 for l in all_label_ints if l == cls_idx)
    print(f"   {cls_name:<8}: {count}")

# ── Стратифицированный сплит 90/10 ────────────────────────────
train_idx, test_idx = train_test_split(
    range(len(all_samples)),
    test_size=0.10,
    stratify=all_label_ints,
    random_state=42
)

train_samples = [all_samples[i] for i in train_idx]
test_samples  = [all_samples[i] for i in test_idx]

# ── Трансформации ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Создаём датасеты и лоадеры ─────────────────────────────────
train_dataset = WildlifeDatasetFromJSON(train_samples, DATASET_DIR, train_transform)
test_dataset  = WildlifeDatasetFromJSON(test_samples,  DATASET_DIR, test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Статистика ─────────────────────────────────────────────────
print(f"\n✅ Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print()
for cls_name, cls_idx in CLASSES.items():
    tr = sum(1 for _, l in train_dataset.samples if l == cls_idx)
    te = sum(1 for _, l in test_dataset.samples  if l == cls_idx)
    print(f"   {cls_name:<8}  train={tr:>5}  test={te:>4}")


Построение модели ResNet-50

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    # Загружаем предобученный ResNet-50 (веса ImageNet)
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Размораживаем только последние 2 ResNet-блока + FC
    # Это быстрее, чем полный fine-tuning, и снижает риск переобучения
    for name, param in model.named_parameters():
        if "layer3" in name or "layer4" in name or "fc" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    # Заменяем классификационную голову на нашу (3 класса)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )

    return model.to(DEVICE)

model = build_model(NUM_CLASSES)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🔧 Параметров к обучению: {trainable:,} / {total:,}")

Обучение

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, momentum=0.9, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

train_losses, train_accs = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, predicted = outputs.detach().max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc  = correct / total * 100
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    # ✅ БЫЛО: if epoch % 5 == 0 or epoch == 1 — печатало только каждую 5-ю
    # ✅ СТАЛО: печатаем каждую эпоху
    print(f"Epoch [{epoch:>3}/{EPOCHS}]  "
          f"Loss: {epoch_loss:.4f}  "
          f"Acc: {epoch_acc:.2f}%  "
          f"LR: {scheduler.get_last_lr()[0]:.5f}")

torch.save(model.state_dict(), MODEL_SAVE)
print(f"\n✅ Модель сохранена: {MODEL_SAVE}")


График обучения

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(train_losses, color="crimson")
ax1.set_title("Loss по эпохам")
ax1.set_xlabel("Эпоха")
ax1.set_ylabel("CrossEntropy Loss")
ax1.grid(True)

ax2.plot(train_accs, color="steelblue")
ax2.set_title("Accuracy по эпохам")
ax2.set_xlabel("Эпоха")
ax2.set_ylabel("Accuracy, %")
ax2.grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("📊 График сохранён: training_curves.png")

Метрики на обучающей выборке

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize

model.eval()
all_preds, all_labels_test, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="📊 Оценка на тест-сете"):
        imgs = imgs.to(DEVICE)
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_preds.extend(preds)
        all_labels_test.extend(labels.numpy())
        all_probs.extend(probs)

all_preds       = np.array(all_preds)
all_labels_test = np.array(all_labels_test)
all_probs       = np.array(all_probs)
class_names     = [IDX2CLASS[i] for i in range(NUM_CLASSES)]

print("=" * 60)
print("📋 Classification Report (Тестовая выборка)")
print("=" * 60)
print(classification_report(all_labels_test, all_preds,
                            target_names=class_names, digits=4))

# ── Метрики по каждому классу ──────────────────────────────────
print("=" * 60)
print("📐 Детальные метрики по классам")
print("=" * 60)

# Бинаризация меток для подсчёта specificity и ROC-AUC
labels_bin = label_binarize(all_labels_test, classes=list(range(NUM_CLASSES)))

for i, cls in enumerate(class_names):
    tp = np.sum((all_preds == i) & (all_labels_test == i))
    tn = np.sum((all_preds != i) & (all_labels_test != i))
    fp = np.sum((all_preds == i) & (all_labels_test != i))
    fn = np.sum((all_preds != i) & (all_labels_test == i))

    precision   = tp / (tp + fp + 1e-9)
    recall      = tp / (tp + fn + 1e-9)   # = Sensitivity
    specificity = tn / (tn + fp + 1e-9)
    f1          = 2 * precision * recall / (precision + recall + 1e-9)
    roc_auc     = roc_auc_score(labels_bin[:, i], all_probs[:, i])

    print(f"\n  [{cls.upper()}]")
    print(f"    Precision:   {precision:.4f}")
    print(f"    Recall:      {recall:.4f}")
    print(f"    Specificity: {specificity:.4f}")
    print(f"    F1:          {f1:.4f}")
    print(f"    ROC-AUC:     {roc_auc:.4f}")

# ── Сводные метрики ────────────────────────────────────────────
f1_macro   = f1_score(all_labels_test, all_preds, average="macro")
roc_macro  = roc_auc_score(labels_bin, all_probs,
                           multi_class="ovr", average="macro")

print("\n" + "=" * 60)
print(f"  F1 macro  (3 класса):   {f1_macro:.4f}")
print(f"  ROC-AUC macro (OvR):    {roc_macro:.4f}")
print("=" * 60)

# ── Матрица ошибок ─────────────────────────────────────────────
cm = confusion_matrix(all_labels_test, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title("Матрица ошибок (Тестовая выборка)", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# ── Попарная матрица лиса vs волк (требование ТЗ) ──────────────
fw_mask  = np.isin(all_labels_test, [CLASSES["fox"], CLASSES["wolf"]])
cm_fw = confusion_matrix(
    all_labels_test[fw_mask],
    all_preds[fw_mask],
    labels=[CLASSES["fox"], CLASSES["wolf"]]
)
fig2, ax2 = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_fw, display_labels=["fox", "wolf"]).plot(
    ax=ax2, colorbar=False, cmap="Oranges"
)
ax2.set_title("Матрица ошибок: лиса vs волк", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrix_fox_wolf.png", dpi=150)
plt.show()
print("✅ Матрицы сохранены")


Инференс на отдельной папке (тест)

In [ ]:
import pandas as pd

rows = []
for i, cls in enumerate(class_names):
    tp = np.sum((all_preds == i) & (all_labels_test == i))
    tn = np.sum((all_preds != i) & (all_labels_test != i))
    fp = np.sum((all_preds == i) & (all_labels_test != i))
    fn = np.sum((all_preds != i) & (all_labels_test == i))

    rows.append({
        "class":       cls,
        "precision":   round(tp / (tp + fp + 1e-9), 4),
        "recall":      round(tp / (tp + fn + 1e-9), 4),
        "specificity": round(tn / (tn + fp + 1e-9), 4),
        "f1":          round(2*tp / (2*tp + fp + fn + 1e-9), 4),
        "roc_auc":     round(roc_auc_score(labels_bin[:, i], all_probs[:, i]), 4),
    })

rows.append({
    "class":       "MACRO",
    "precision":   round(precision_score(all_labels_test, all_preds, average="macro"), 4),
    "recall":      round(recall_score(all_labels_test, all_preds, average="macro"), 4),
    "specificity": "-",
    "f1":          round(f1_macro, 4),
    "roc_auc":     round(roc_macro, 4),
})

df_metrics = pd.DataFrame(rows)
df_metrics.to_csv("metrics.csv", index=False)
print(df_metrics.to_string(index=False))
print("\n✅ Метрики сохранены в metrics.csv")
